In [1]:
from datasets import load_dataset

from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

## Load & Preprocess

In [2]:
id2label = {0: "Win A", 1: "Tie", 2: "Win B"}
label2id = {v:k for k, v in id2label.items()}

model_string = "distilbert/distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_string, num_labels=3, id2label=id2label, label2id=label2id
)
tokenizer = AutoTokenizer.from_pretrained(model_string)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
# limit dataset size for now
dataset = load_dataset("csv", data_files="data/train.csv")["train"].select(range(16000))

In [26]:
def preprocess_input(batch):
    """Unifies the prompts and responses into a single sentence and creates a single integer label from the label columns."""
    prompts = []
    for prompt, resp_a, resp_b in zip(batch["prompt"], batch["response_a"], batch["response_b"]):
        prompts.append(f"<prompt>{prompt}\n\n<answer1>{resp_a}\n\n<answer2>{resp_b}")

    labels = []
    for win_a, win_b in zip(batch["winner_model_a"], batch["winner_model_b"]):
        label = 0 if win_a else 2 if win_b else 1
        labels.append(label)
    return {**tokenizer(prompts, truncation=True), "labels": labels}

In [27]:
preprocessed = dataset.map(preprocess_input, batched=True)
dataset_split = preprocessed.train_test_split(test_size=0.1) # shuffled by default

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

In [28]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Training

In [31]:
training_args = TrainingArguments(
    output_dir=f"models/{model_string}",
    logging_dir="tensor-runs/test",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="tensorboard"     # log to tensorboard
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_split["train"],
    eval_dataset=dataset_split["test"],
    # processing_class=tokenizer,
    data_collator=data_collator
)

In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,1.049056


TrainOutput(global_step=450, training_loss=1.007738986545139, metrics={'train_runtime': 105.5604, 'train_samples_per_second': 136.415, 'train_steps_per_second': 4.263, 'total_flos': 1907564558745600.0, 'train_loss': 1.007738986545139, 'epoch': 1.0})

## Inference

In [1]:
import pandas as pd
from transformers import Pipeline
from transformers.pipelines.pt_utils import KeyDataset

In [4]:
# the model gets loaded automatically after train
# only call this if no training was conducted
model = AutoModelForSequenceClassification.from_pretrained("distilbert/checkpoint-3234/")

tokenizer = AutoTokenizer.from_pretrained("distilbert/checkpoint-3234/")

In [5]:
def preprocess_for_inf(batch):
    """Unifies the prompts and responses into a single sentence and creates a single integer label from the label columns."""
    prompts = []
    for prompt, resp_a, resp_b in zip(batch["prompt"], batch["response_a"], batch["response_b"]):
        prompts.append(f"<prompt>{prompt}\n\n<answer1>{resp_a}\n\n<answer2>{resp_b}")

    return {**tokenizer(prompts, truncation=True), "text": prompts}

In [6]:
test_data = load_dataset("csv", data_files={"test": "data/test.csv"})["test"]
test_data = test_data.map(preprocess_for_inf, batched=True)

In [7]:
class SoftmaxPipeline(Pipeline):
    def preprocess(self, inputs, **kwargs):
        return self.tokenizer(inputs, return_tensors="pt", truncation=True)

    def _forward(self, model_inputs, **kwargs):
        outputs = self.model(**model_inputs)
        return outputs

    def postprocess(self, model_outputs, **kwargs):
        return model_outputs["logits"].softmax(dim=-1)

    def _sanitize_parameters(self, **kwargs):
        return {}, {}, {}

pipe = SoftmaxPipeline(task="text-classification", model=model, tokenizer=tokenizer, batch_size=4)

Device set to use cuda:0


In [11]:
from collections import defaultdict

test_result = defaultdict(list)
for sample_id, out in zip(test_data["id"], pipe(KeyDataset(test_data, "text"), batch_size=8)):
    probs = out[0].tolist()
    test_result["id"].append(sample_id)
    test_result["winner_model_a"].append(probs[0])
    test_result["winner_model_b"].append(probs[2])
    test_result["tie"].append(probs[1])

test_result = pd.DataFrame(test_result)
test_result.to_csv("submission.csv", index=False)